# Evaluate Release Assessments

Runs deterministic assessments across all bundled scenarios, inspects score composition,
and visualises the decision breakdown per sample.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

from app.services.sample_data import SampleDataRepository
from app.services.assessment_service import AssessmentService
from app.services.rules_engine import RulesEngine
from app.services.decision_policy import DecisionPolicy
from app.providers.mock_provider import MockMemoProvider

## 1 — Run All Sample Fixtures

Create a standalone AssessmentService (no DB, no OpenAI) and assess every sample bundle.

In [ ]:
sample_repo = SampleDataRepository()
service = AssessmentService(
    rules_engine=RulesEngine(),
    policy=DecisionPolicy(),
    memo_provider=MockMemoProvider(),
)

assessments = {}
for name in sample_repo.list_samples():
    bundle = sample_repo.load_sample(name)
    assessments[name] = service.assess(bundle)
    print(f'{name}: decision={assessments[name].decision.decision.value}')

## 2 — Summary Table

Compare decision, risk score, evidence coverage, block and flag counts across scenarios.

In [ ]:
header = f"{'Sample':<30} {'Decision':<10} {'Score':>6} {'Coverage':>9} {'Blocks':>7} {'Flags':>6}"
print(header)
print('-' * len(header))
for name, a in sorted(assessments.items(), key=lambda kv: kv[1].rules.risk_score, reverse=True):
    print(f"{name:<30} {a.decision.decision.value:<10} {a.rules.risk_score:>6.1f} "
          f"{a.rules.evidence_coverage:>8.0%} {len(a.rules.hard_blocks):>7} {len(a.rules.risk_flags):>6}")

## 3 — Hard Blocks Detail

Show every hard-block finding for each assessed sample.

In [ ]:
for name, a in assessments.items():
    if a.rules.hard_blocks:
        print(f'\n=== {name} ===')
        for block in a.rules.hard_blocks:
            print(f'  [{block.rule_id}] {block.message} (weight={block.weight})')
    else:
        print(f'\n=== {name} === (no hard blocks)')

## 4 — Risk Flags Breakdown

Show weighted risk flags that contributed to the score.

In [ ]:
for name, a in assessments.items():
    if a.rules.risk_flags:
        print(f'\n=== {name} (score={a.rules.risk_score}) ===')
        for flag in sorted(a.rules.risk_flags, key=lambda f: f.weight, reverse=True):
            print(f'  {flag.weight:>5.1f}  {flag.rule_id:<30} {flag.message}')
    else:
        print(f'\n=== {name} === (no risk flags)')

## 5 — Evidence Coverage per Category

Show which evidence categories are present or missing for each sample.

In [ ]:
for name, a in assessments.items():
    print(f'\n=== {name} (overall: {a.rules.evidence_coverage:.0%}) ===')
    for cat, score in a.rules.coverage_by_category.items():
        status = 'PRESENT' if score >= 1.0 else 'MISSING'
        print(f'  {cat:<25} {status}')

## 6 — Policy Decision Rationale

Print the policy rationale and any coverage downgrade for each scenario.

In [ ]:
for name, a in assessments.items():
    d = a.decision
    print(f'\n=== {name} ===')
    print(f'  Decision : {d.decision.value}')
    print(f'  Base     : {d.base_decision.value}')
    print(f'  Downgrade: {d.downgraded_for_coverage}')
    print(f'  Rationale: {d.rationale}')
    print(f'  Triggers : {d.triggered_conditions}')

## 7 — Mock Memo Output

Show the deterministic memo generated for each scenario.

In [ ]:
for name, a in assessments.items():
    print(f'\n=== {name} ===')
    print(f'  Summary: {a.memo.summary}')
    print(f'  Top risks: {a.memo.top_risks}')
    print(f'  Next steps: {a.memo.next_steps}')